# Notebook 13 — Recovery Time and Resilience

This notebook extends Notebook 12 from **failure-boundary detection** to **post-failure resilience**.

Core question:

> When CGCS phase-lock drops below the 45° threshold, which control policy recovers fastest?

Outputs:

- figures in `figures/recovery_time_and_resilience/`
- results in `results/recovery_time_and_resilience_summary.json`
- summary markdown in `results/recovery_time_and_resilience_summary.md`
- docs markdown in `docs/recovery_time_and_resilience.md`


In [ ]:
import os
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(13)

NOTEBOOK_ID = "13"
SLUG = "recovery_time_and_resilience"
FIG_DIR = Path("figures") / SLUG
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

THRESHOLD = 1 / np.sqrt(1**2 + 1**2)
N_BLOCKS = 80
blocks = np.arange(N_BLOCKS)

print("CGCS threshold:", THRESHOLD)
print("Figure dir:", FIG_DIR)


## 1. Shared disturbance simulation

Notebook 13 uses the same control-stack stress idea as Notebook 12:

- smooth baseline drift
- noise-burst window
- model-mismatch window
- shock/outlier events

The goal is not perfect hardware realism. The goal is a repeatable control-stack benchmark for recovery metrics.

In [ ]:
# Disturbance windows
noise_burst = (28, 40)
model_mismatch = (52, 66)
shock_blocks = np.array([26, 38, 52, 69, 74])

# Baseline parameter drift
omega_true = (
    0.22 * np.sin(0.23 * blocks)
    + 0.17 * np.sin(0.61 * blocks + 0.4)
    + 0.006 * blocks / N_BLOCKS
)
B_true = (
    0.006 * np.sin(0.45 * blocks)
    + 0.0010 * blocks
    + 0.014 / (1 + np.exp(-(blocks - 44) / 4))
)

# Measurement noise, intentionally amplified during burst/mismatch windows
omega_noise_scale = np.full(N_BLOCKS, 0.020)
B_noise_scale = np.full(N_BLOCKS, 0.006)
omega_noise_scale[noise_burst[0]:noise_burst[1]+1] *= 2.5
B_noise_scale[noise_burst[0]:noise_burst[1]+1] *= 3.0
omega_noise_scale[model_mismatch[0]:model_mismatch[1]+1] *= 2.0
B_noise_scale[model_mismatch[0]:model_mismatch[1]+1] *= 2.2

omega_meas = omega_true + np.random.normal(0, omega_noise_scale)
B_meas = B_true + np.random.normal(0, B_noise_scale)

# Shock events: large measurement spikes
omega_meas[shock_blocks] += np.array([2.8, -0.9, 0.3, 1.1, -0.7])
B_meas[shock_blocks] += np.array([0.40, 0.28, 0.26, -0.38, 0.33])

# Model mismatch: true response drifts while measurements remain deceptive
omega_true[model_mismatch[0]:model_mismatch[1]+1] += np.linspace(-0.22, 0.70, model_mismatch[1]-model_mismatch[0]+1)
B_true[model_mismatch[0]:model_mismatch[1]+1] += np.linspace(0.03, -0.04, model_mismatch[1]-model_mismatch[0]+1)

plt.figure(figsize=(12, 4))
plt.plot(blocks, omega_true, label="true Ω drift")
plt.plot(blocks, omega_meas, "o-", alpha=0.7, label="measured Ω")
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
plt.scatter(shock_blocks, omega_meas[shock_blocks], marker="x", s=80, label="shock blocks")
plt.axhline(0)
plt.title("Recovery benchmark: Ω disturbance profile")
plt.xlabel("calibration block")
plt.ylabel("Ω drift / measurement")
plt.legend()
plt.show()


## 2. Controller / estimator policies

Policies intentionally mirror the control-stack notebooks:

- `none`
- `moving_average`
- `scalar_kalman`
- `joint_kalman`
- `robust_gated_kalman`
- `constrained_mpc`
- `oracle`

The policies produce compensation commands for Ω and B, then response curves are compared against the oracle target.

In [ ]:
def moving_average(x, window=5):
    out = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        out[i] = np.mean(x[lo:i+1])
    return out


def scalar_kalman(z, q=4e-4, r=1.5e-3, x0=None):
    x = z[0] if x0 is None else x0
    p = 1.0
    xs = []
    for zi in z:
        p = p + q
        k = p / (p + r)
        x = x + k * (zi - x)
        p = (1 - k) * p
        xs.append(x)
    return np.array(xs)


def joint_kalman(z_omega, z_B, q_omega=7e-4, q_B=2e-5, q_cross=2.5e-5, r_omega=1.5e-3, r_B=5e-5):
    z = np.vstack([z_omega, z_B]).T
    x = z[0].copy()
    P = np.eye(2) * 0.05
    Q = np.array([[q_omega, q_cross], [q_cross, q_B]])
    R = np.diag([r_omega, r_B])
    H = np.eye(2)
    xs = []
    for zi in z:
        P = P + Q
        S = H @ P @ H.T + R
        K = P @ H.T @ np.linalg.inv(S)
        x = x + K @ (zi - H @ x)
        P = (np.eye(2) - K @ H) @ P
        xs.append(x.copy())
    return np.array(xs)


def robust_gated_joint(z_omega, z_B, gate_omega=0.42, gate_B=0.11):
    # Gate large innovations before joint update.
    z = np.vstack([z_omega, z_B]).T
    x = z[0].copy()
    P = np.eye(2) * 0.05
    Q = np.array([[7e-4, 2.5e-5], [2.5e-5, 2e-5]])
    R = np.diag([1.5e-3, 5e-5])
    H = np.eye(2)
    xs = []
    for zi in z:
        P = P + Q
        innov = zi - x
        zi_gated = zi.copy()
        if abs(innov[0]) > gate_omega:
            zi_gated[0] = x[0] + np.sign(innov[0]) * gate_omega
        if abs(innov[1]) > gate_B:
            zi_gated[1] = x[1] + np.sign(innov[1]) * gate_B
        S = H @ P @ H.T + R
        K = P @ H.T @ np.linalg.inv(S)
        x = x + K @ (zi_gated - H @ x)
        P = (np.eye(2) - K @ H) @ P
        xs.append(x.copy())
    return np.array(xs)


def constrained_mpc_like(omega_est, B_est, eps_omega=0.08, eps_B=0.035, alpha=0.72):
    # Smooth and command-bound an estimator trajectory.
    omega_cmd = np.zeros_like(omega_est)
    B_cmd = np.zeros_like(B_est)
    omega_cmd[0] = omega_est[0]
    B_cmd[0] = B_est[0]
    for i in range(1, len(omega_est)):
        desired_o = alpha * omega_est[i] + (1-alpha) * omega_est[i-1]
        desired_b = alpha * B_est[i] + (1-alpha) * B_est[i-1]
        omega_cmd[i] = omega_cmd[i-1] + np.clip(desired_o - omega_cmd[i-1], -eps_omega, eps_omega)
        B_cmd[i] = B_cmd[i-1] + np.clip(desired_b - B_cmd[i-1], -eps_B, eps_B)
    return omega_cmd, B_cmd

ma_o = moving_average(omega_meas, 5)
ma_b = moving_average(B_meas, 5)
sc_o = scalar_kalman(omega_meas, q=6e-4, r=1.0e-3)
sc_b = scalar_kalman(B_meas, q=2e-5, r=5e-5)
joint = joint_kalman(omega_meas, B_meas)
robust = robust_gated_joint(omega_meas, B_meas)
cmpc_o, cmpc_b = constrained_mpc_like(robust[:,0], robust[:,1])

policies = {
    "none": (np.zeros(N_BLOCKS), np.zeros(N_BLOCKS)),
    "moving_average": (ma_o, ma_b),
    "scalar_kalman": (sc_o, sc_b),
    "joint_kalman": (joint[:,0], joint[:,1]),
    "robust_gated_kalman": (robust[:,0], robust[:,1]),
    "constrained_mpc": (cmpc_o, cmpc_b),
    "oracle": (omega_true, B_true),
}

print("policies:", list(policies.keys()))


## 3. Response model and CGCS cosine

For each calibration block, the controller command determines an excited-state response curve.

The CGCS score is cosine similarity between controlled response and target response.

In [ ]:
t = np.linspace(0, 10, 450)
base_omega = 1.0
base_B = 0.0


def response_curve(omega_cmd, B_cmd):
    # Probability-like response. Clipped for stability under severe mismatch.
    y = np.sin((base_omega + omega_cmd) * t / 2) ** 2 + B_cmd
    return np.clip(y, 0, 1.05)


def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

responses = {}
cosines = {}
rmse_series = {}
threshold_margins = {}

target_responses = [response_curve(omega_true[i], B_true[i]) for i in range(N_BLOCKS)]

for name, (ocmd, bcmd) in policies.items():
    ys = []
    cs = []
    rmses = []
    for i in range(N_BLOCKS):
        y = response_curve(ocmd[i], bcmd[i])
        target = target_responses[i]
        ys.append(y)
        cs.append(cosine_similarity(y, target))
        rmses.append(np.sqrt(np.mean((y - target)**2)))
    responses[name] = ys
    cosines[name] = np.array(cs)
    rmse_series[name] = np.array(rmses)
    threshold_margins[name] = np.array(cs) - THRESHOLD

summary_rows = []
for name in policies:
    summary_rows.append({
        "policy": name,
        "mean_rmse": float(np.mean(rmse_series[name])),
        "max_rmse": float(np.max(rmse_series[name])),
        "min_cosine": float(np.min(cosines[name])),
        "mean_cosine": float(np.mean(cosines[name])),
    })

pd.DataFrame(summary_rows).sort_values("mean_rmse")


## 4. Recovery-time metric

A failure starts when:

```text
cosθ < 1 / √(1² + 1²)
```

Recovery time is measured as the number of calibration blocks until the same policy returns above threshold. Consecutive below-threshold blocks count as one failure episode.

In [ ]:
def failure_episodes(cos_values, threshold=THRESHOLD):
    below = cos_values < threshold
    episodes = []
    i = 0
    n = len(below)
    while i < n:
        if below[i]:
            start = i
            while i < n and below[i]:
                i += 1
            end = i - 1
            recovered_at = i if i < n else None
            recovery_time = (recovered_at - start) if recovered_at is not None else (n - start)
            episodes.append({
                "start": start,
                "end": end,
                "duration_below": end - start + 1,
                "recovered_at": recovered_at,
                "recovery_time": recovery_time,
            })
        else:
            i += 1
    return episodes

recovery_rows = []
episode_rows = []
for name in policies:
    episodes = failure_episodes(cosines[name])
    durations = [e["duration_below"] for e in episodes]
    recoveries = [e["recovery_time"] for e in episodes]
    time_below = int(np.sum(cosines[name] < THRESHOLD))
    failure_count = len(episodes)
    mean_recovery = float(np.mean(recoveries)) if recoveries else 0.0
    max_recovery = float(np.max(recoveries)) if recoveries else 0.0
    margin_mean = float(np.mean(threshold_margins[name]))
    resilience_score = margin_mean - 0.10 * failure_count - 0.05 * mean_recovery - 0.01 * time_below
    recovery_rows.append({
        "policy": name,
        "failure_count": int(failure_count),
        "time_below_threshold": int(time_below),
        "mean_recovery_time": mean_recovery,
        "max_recovery_time": max_recovery,
        "mean_threshold_margin": margin_mean,
        "resilience_score": float(resilience_score),
        "mean_rmse": float(np.mean(rmse_series[name])),
        "max_rmse": float(np.max(rmse_series[name])),
    })
    for e in episodes:
        episode_rows.append({"policy": name, **e})

recovery_df = pd.DataFrame(recovery_rows).sort_values(["failure_count", "time_below_threshold", "mean_rmse"])
episodes_df = pd.DataFrame(episode_rows)
recovery_df


## 5. Export helpers

In [ ]:
def save_fig(filename):
    png = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{filename}.png"
    pdf = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{filename}.pdf"
    plt.savefig(png, dpi=220, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print("saved", png)
    print("saved", pdf)


## 6. Figures

In [ ]:
plt.figure(figsize=(13, 5))
for name in policies:
    plt.plot(blocks, cosines[name], marker="o", label=name)
plt.axhline(THRESHOLD, linestyle="--", label="45° threshold")
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
plt.title("Recovery time: CGCS phase-lock stability")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.55, 1.01)
plt.legend(loc="lower left", ncol=1)
save_fig("cgcs_stability_comparison")
plt.show()


In [ ]:
plt.figure(figsize=(13, 5))
for name in policies:
    if name == "oracle":
        continue
    plt.plot(blocks, threshold_margins[name], marker="o", label=name)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
plt.title("Recovery time: threshold margin")
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.legend(loc="lower left")
save_fig("threshold_margin")
plt.show()


In [ ]:
failure_matrix = np.vstack([(cosines[name] < THRESHOLD).astype(int) for name in policies])
plt.figure(figsize=(13, 4.5))
plt.imshow(failure_matrix, aspect="auto", interpolation="nearest")
plt.yticks(np.arange(len(policies)), list(policies.keys()))
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.title("Recovery time: failure-event map")
plt.colorbar(label="failure: cosθ < threshold")
save_fig("failure_events")
plt.show()


In [ ]:
plot_df = recovery_df[recovery_df.policy != "oracle"].copy()
plt.figure(figsize=(11, 5))
x = np.arange(len(plot_df))
plt.bar(x - 0.2, plot_df["mean_recovery_time"], width=0.4, label="mean recovery time")
plt.bar(x + 0.2, plot_df["max_recovery_time"], width=0.4, label="max recovery time")
plt.xticks(x, plot_df["policy"], rotation=25, ha="right")
plt.ylabel("blocks")
plt.title("Recovery time: recovery-time bars")
plt.legend()
save_fig("recovery_time_bars")
plt.show()


In [ ]:
plot_df = recovery_df[recovery_df.policy != "oracle"].copy()
plt.figure(figsize=(11, 5))
plt.bar(plot_df["policy"], plot_df["time_below_threshold"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("blocks below threshold")
plt.title("Recovery time: total time below threshold")
save_fig("time_below_threshold")
plt.show()


In [ ]:
rank_df = recovery_df.sort_values("resilience_score", ascending=False).copy()
plt.figure(figsize=(12, 5))
x = np.arange(len(rank_df))
plt.bar(x, rank_df["resilience_score"])
plt.xticks(x, rank_df["policy"], rotation=25, ha="right")
plt.ylabel("resilience score")
plt.title("Recovery time: resilience ranking summary")
save_fig("resilience_ranking_summary")
plt.show()


In [ ]:
# Find worst recovery episode among non-oracle policies.
if not episodes_df.empty:
    worst = episodes_df[episodes_df.policy != "oracle"].sort_values("recovery_time", ascending=False).iloc[0]
    start = max(0, int(worst.start) - 5)
    end = min(N_BLOCKS-1, int(worst.end) + 8)
else:
    worst = None
    start, end = 20, 45

plt.figure(figsize=(13, 5))
for name in policies:
    if name == "oracle":
        continue
    plt.plot(blocks[start:end+1], cosines[name][start:end+1], marker="o", label=name)
plt.axhline(THRESHOLD, linestyle="--", label="45° threshold")
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
plt.title(f"Recovery time: worst recovery window ({start}–{end})")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.legend(loc="lower left")
save_fig("worst_recovery_window")
plt.show()


In [ ]:
zoom_start, zoom_end = noise_burst[0]-5, model_mismatch[1]+5
plt.figure(figsize=(13, 5))
for name in ["scalar_kalman", "joint_kalman", "robust_gated_kalman", "constrained_mpc"]:
    plt.plot(blocks[zoom_start:zoom_end+1], rmse_series[name][zoom_start:zoom_end+1], marker="o", label=name)
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
plt.title("Recovery time: disturbance-window zoom")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.legend()
save_fig("disturbance_window_zoom")
plt.show()


## 7. Results JSON and markdown exports

In [ ]:
summary = {
    "notebook": f"{NOTEBOOK_ID}_{SLUG}",
    "threshold": float(THRESHOLD),
    "n_blocks": int(N_BLOCKS),
    "noise_burst_window": list(noise_burst),
    "model_mismatch_window": list(model_mismatch),
    "shock_blocks": shock_blocks.tolist(),
    "policy_metrics": recovery_df.to_dict(orient="records"),
    "failure_episodes": episodes_df.to_dict(orient="records") if not episodes_df.empty else [],
    "figures": [
        f"{NOTEBOOK_ID}_{SLUG}_cgcs_stability_comparison.png",
        f"{NOTEBOOK_ID}_{SLUG}_threshold_margin.png",
        f"{NOTEBOOK_ID}_{SLUG}_failure_events.png",
        f"{NOTEBOOK_ID}_{SLUG}_recovery_time_bars.png",
        f"{NOTEBOOK_ID}_{SLUG}_time_below_threshold.png",
        f"{NOTEBOOK_ID}_{SLUG}_resilience_ranking_summary.png",
        f"{NOTEBOOK_ID}_{SLUG}_worst_recovery_window.png",
        f"{NOTEBOOK_ID}_{SLUG}_disturbance_window_zoom.png",
    ],
}

json_path = RESULTS_DIR / f"{SLUG}_summary.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)
print("saved", json_path)

# Build markdown tables
metric_cols = ["policy", "failure_count", "time_below_threshold", "mean_recovery_time", "max_recovery_time", "mean_rmse", "max_rmse", "resilience_score"]
md_table = recovery_df[metric_cols].to_markdown(index=False, floatfmt=".5f")

results_md = f"""# Recovery Time and Resilience Results Summary

## Configuration

- Blocks: {N_BLOCKS}
- CGCS threshold: {THRESHOLD:.4f}
- Noise burst window: blocks {noise_burst[0]}–{noise_burst[1]}
- Model mismatch window: blocks {model_mismatch[0]}–{model_mismatch[1]}
- Shock blocks: {', '.join(map(str, shock_blocks))}

---

## Recovery Metrics

{md_table}

---

## Key Observations

- Notebook 12 identifies where phase-lock failure happens; Notebook 13 measures how quickly each policy recovers.
- Kalman policies usually minimize recovery duration because they reconstruct state rather than only smoothing measurements.
- Robust gated Kalman is designed to reduce outlier damage, but aggressive gating can delay recovery if measurements remain informative.
- Constrained MPC can smooth command behavior but may recover slower when command bounds restrict correction.
- No-control and moving-average policies spend more time below threshold when phase alignment drifts.

---

## Conclusion

Recovery time is a stronger resilience metric than mean RMSE alone. A control policy is useful when it not only reduces error, but also returns the system above the CGCS threshold after disturbance.
"""

results_md_path = RESULTS_DIR / f"{SLUG}_summary.md"
results_md_path.write_text(results_md)
print("saved", results_md_path)

figure_sections = "

".join([
    f"### {fig.replace(f'{NOTEBOOK_ID}_{SLUG}_', '').replace('.png','').replace('_',' ').title()}

![{fig}](../figures/{SLUG}/{fig})"
    for fig in summary["figures"]
])

docs_md = f"""# Recovery Time and Resilience (Notebook 13)

Notebook 13 extends the failure-boundary study by measuring recovery after CGCS phase-lock failure.

## Core metric

A failure occurs when:

```text
cosθ < 1 / √(1² + 1²)
```

Recovery time measures how many calibration blocks a policy takes to return above the 45° CGCS threshold.

## Figures

{figure_sections}

## Results

See:

```text
results/{SLUG}_summary.json
results/{SLUG}_summary.md
```

## Takeaway

Notebook 13 reframes control performance as resilience: not only whether a controller reduces RMSE, but whether it restores phase-lock after disturbance.
"""

docs_md_path = DOCS_DIR / f"{SLUG}.md"
docs_md_path.write_text(docs_md)
print("saved", docs_md_path)


## 8. Next notebook direction

Notebook 14 should convert recovery diagnostics into **early-warning prediction**:

- innovation variance
- threshold-margin derivative
- pre-failure warning score
- lead time before CGCS breach
- warning precision/recall
